**© Copyright AIDENTIFY. All rights reserved.**

# Part 3 | Session 20: DeepSeek R1 Case Study — 분석/이해 전용

---

### 📋 학습 목표

본 세션은 **DeepSeek R1 학습 파이프라인을 분석·이해**합니다. 실제 GRPO 학습 코드는 **Session 21**에서 진행합니다.

- 🔹 DeepSeek R1의 4단계 학습 파이프라인(Cold Start → Reasoning RL → RS+SFT → Final RL) 해부
- 🔹 R1에서 보고된 "Aha Moment" / 자발적 CoT 발현 현상 이해
- 🔹 GRPO 알고리즘의 핵심 아이디어와 PPO와의 차이
- 🔹 Rule-based reward 함수 설계 원리 (정확성/형식 보상)

### 📦 필요 라이브러리

```
(없음 — 본 세션은 분석 텍스트와 시각화만 다룹니다)
```

### ⏱️ 예상 소요 시간: 약 40분

### 🔗 학습 흐름
- 선행: **Session 17** RL 개념, **Session 19** DPO 실습
- 후속: **Session 20b** Rejection Sampling 실습 (R1의 RS+SFT 단계)
- 후속: **Session 21** GRPO 실습 (R1의 Reasoning RL 단계)

> 💡 본 노트북은 "왜 GRPO인가? R1은 어떻게 학습됐나?"를 다루는 **개념 노트북**입니다.
> GRPOTrainer 실행 코드는 **Session 21에서 통합 실습**합니다.

---

In [ ]:
# 본 노트북은 분석 위주 — GPU/학습 라이브러리 불필요
import json
print("✅ Session 20 시작 — DeepSeek R1 학습 파이프라인 분석")

---
## 1️⃣ DeepSeek R1의 학습 과정 분석

### DeepSeek R1이란?

DeepSeek R1은 2024년 말 DeepSeek에서 발표한 **추론 특화 LLM**으로, OpenAI의 o1/o3에 필적하는 성능을 보여주었습니다.

### 핵심 혁신

```
🔑 핵심: 강화학습(GRPO)만으로 추론 능력을 획득!

기존 접근: SFT로 CoT 데이터 학습 → 추론 패턴 모방
DeepSeek R1: RL(GRPO)로 직접 추론 능력 발현 → 자발적 CoT 출현
```

### DeepSeek R1 학습 4단계

```
┌──────────────────────────────────────────────┐
│          DeepSeek R1 학습 파이프라인            │
├──────────────────────────────────────────────┤
│                                              │
│  Stage 1: Cold Start (소량 SFT)               │
│  └→ 기본적인 CoT 형식 학습                      │
│                                              │
│  Stage 2: 추론 중심 RL (GRPO)                  │
│  └→ 수학/코딩 문제로 추론 능력 강화              │
│  └→ 보상: 정답 여부 + 형식 보상                 │
│                                              │
│  Stage 3: Rejection Sampling + SFT            │
│  └→ RL 모델의 좋은 출력을 수집하여 재학습         │
│  └→ 일반 능력(글쓰기, 번역 등) 데이터 혼합       │
│                                              │
│  Stage 4: 전체 RL (GRPO)                      │
│  └→ 추론 + 일반 태스크 모두 보상 부여            │
│  └→ 안전성/유용성 보상 추가                     │
│                                              │
│  최종: 추론 + 일반 능력 모두 갖춘 모델            │
└──────────────────────────────────────────────┘
```

In [2]:
# DeepSeek R1 학습 단계 상세 분석
print("=" * 60)
print("📌 DeepSeek R1 학습 파이프라인 분석")
print("=" * 60)

stages = [
    {
        "name": "Stage 1: Cold Start SFT",
        "description": "소량의 CoT 데이터로 기본 형식 학습",
        "details": [
            "수천 개의 <think>...</think> 형식 예시 학습",
            "기본적인 사고 과정 표현 방법 습득",
            "목적: RL이 탐색할 수 있는 기본 능력 확보",
        ]
    },
    {
        "name": "Stage 2: 추론 중심 RL (GRPO)",
        "description": "수학/코딩 문제로 추론 능력 강화",
        "details": [
            "GRPO 알고리즘 사용 (Value Model 불필요)",
            "보상 1: 정답 여부 (정확한 답 = +1, 오답 = 0)",
            "보상 2: 형식 보상 (<think> 태그 사용 여부)",
            "놀라운 발견: 'aha moment' - 모델이 자발적으로 자기 검증 학습!",
        ]
    },
    {
        "name": "Stage 3: Rejection Sampling + SFT",
        "description": "좋은 출력을 수집하여 재학습",
        "details": [
            "Stage 2 모델에서 정답을 맞힌 출력 수집",
            "일반 태스크(글쓰기, 번역, QA) 데이터 혼합",
            "SFT로 추론+일반 능력 통합",
        ]
    },
    {
        "name": "Stage 4: 전체 RL (GRPO)",
        "description": "모든 태스크에 대해 RL 적용",
        "details": [
            "추론 태스크: 정답 기반 보상",
            "일반 태스크: LLM-as-judge 보상",
            "안전성/유용성 보상 추가",
        ]
    },
]

for stage in stages:
    print(f"\n🔷 {stage['name']}")
    print(f"   📋 {stage['description']}")
    for detail in stage['details']:
        print(f"      • {detail}")

📌 DeepSeek R1 학습 파이프라인 분석

🔷 Stage 1: Cold Start SFT
   📋 소량의 CoT 데이터로 기본 형식 학습
      • 수천 개의 <think>...</think> 형식 예시 학습
      • 기본적인 사고 과정 표현 방법 습득
      • 목적: RL이 탐색할 수 있는 기본 능력 확보

🔷 Stage 2: 추론 중심 RL (GRPO)
   📋 수학/코딩 문제로 추론 능력 강화
      • GRPO 알고리즘 사용 (Value Model 불필요)
      • 보상 1: 정답 여부 (정확한 답 = +1, 오답 = 0)
      • 보상 2: 형식 보상 (<think> 태그 사용 여부)
      • 놀라운 발견: 'aha moment' - 모델이 자발적으로 자기 검증 학습!

🔷 Stage 3: Rejection Sampling + SFT
   📋 좋은 출력을 수집하여 재학습
      • Stage 2 모델에서 정답을 맞힌 출력 수집
      • 일반 태스크(글쓰기, 번역, QA) 데이터 혼합
      • SFT로 추론+일반 능력 통합

🔷 Stage 4: 전체 RL (GRPO)
   📋 모든 태스크에 대해 RL 적용
      • 추론 태스크: 정답 기반 보상
      • 일반 태스크: LLM-as-judge 보상
      • 안전성/유용성 보상 추가


In [3]:
# DeepSeek R1의 "Aha Moment" 시연
print("=" * 60)
print("📌 DeepSeek R1의 'Aha Moment' 예시")
print("=" * 60)

print("""
🧠 "Aha Moment"란?
   RL 학습 중 모델이 자발적으로 자기 검증(self-verification) 능력을 획득하는 순간

📝 예시 - 수학 문제 풀이 과정:

<think>
문제: 27 × 14 = ?

풀이 시작:
27 × 14 = 27 × (10 + 4)
        = 27 × 10 + 27 × 4  
        = 270 + 108
        = 378

잠깐, 검증해보자.            ← 🎯 Aha Moment!
27 × 14 = (30 - 3) × 14    ← 다른 방법으로 검증
        = 420 - 42
        = 378 ✓              ← 일치 확인
</think>

27 × 14 = 378

💡 핵심: 이 자기 검증 능력은 SFT로 학습시킨 것이 아니라,
   RL(GRPO)를 통해 자연스럽게 발현된 것입니다!
   
   → 정답을 맞히면 보상을 받으므로, 검증 과정이 정답률을 높이는 데
     도움이 된다는 것을 모델이 스스로 학습한 것입니다.
""")

📌 DeepSeek R1의 'Aha Moment' 예시

🧠 "Aha Moment"란?
   RL 학습 중 모델이 자발적으로 자기 검증(self-verification) 능력을 획득하는 순간

📝 예시 - 수학 문제 풀이 과정:

<think>
문제: 27 × 14 = ?

풀이 시작:
27 × 14 = 27 × (10 + 4)
        = 27 × 10 + 27 × 4  
        = 270 + 108
        = 378

잠깐, 검증해보자.            ← 🎯 Aha Moment!
27 × 14 = (30 - 3) × 14    ← 다른 방법으로 검증
        = 420 - 42
        = 378 ✓              ← 일치 확인
</think>

27 × 14 = 378

💡 핵심: 이 자기 검증 능력은 SFT로 학습시킨 것이 아니라,
   RL(GRPO)를 통해 자연스럽게 발현된 것입니다!

   → 정답을 맞히면 보상을 받으므로, 검증 과정이 정답률을 높이는 데
     도움이 된다는 것을 모델이 스스로 학습한 것입니다.



---
## 2️⃣ GRPO 알고리즘 상세 (Group Relative Policy Optimization)

### GRPO 수학적 정의

```
GRPO 목적 함수:

J(θ) = E_q~P(Q) [ 1/G × Σ_{i=1}^{G} (
    min(
        r_i(θ) × Â_i,
        clip(r_i(θ), 1-ε, 1+ε) × Â_i
    ) - β × KL(π_θ || π_ref)
)]

여기서:
- q: 입력 프롬프트
- G: 그룹 크기 (각 프롬프트당 생성 개수)
- r_i(θ) = π_θ(o_i|q) / π_old(o_i|q): 정책 비율
- Â_i = (r_i - mean(r)) / std(r): 정규화된 이점
- ε: 클리핑 범위
- β: KL 페널티 강도
```

### PPO와의 핵심 차이

```
PPO:  Advantage = Value Function으로 계산 → Value Model 필요
GRPO: Advantage = 그룹 내 상대 비교로 계산 → Value Model 불필요!
```

### GRPO 알고리즘 의사코드

```python
for each batch of prompts Q:
    for each prompt q in Q:
        # 1. G개 응답 생성
        outputs = [generate(q) for _ in range(G)]
        
        # 2. 보상 계산
        rewards = [reward_fn(q, o) for o in outputs]
        
        # 3. 그룹 내 정규화
        advantages = (rewards - mean(rewards)) / std(rewards)
        
    # 4. PPO 스타일 업데이트 (클리핑 + KL 페널티)
    update_policy(outputs, advantages)
```

In [4]:
# GRPO 알고리즘 시뮬레이션
print("=" * 60)
print("📌 GRPO 알고리즘 시뮬레이션")
print("=" * 60)

def simulate_grpo_step(prompt, group_size=4):
    """
    GRPO 한 스텝을 시뮬레이션합니다.
    """
    print(f"\n📝 프롬프트: {prompt['question']}")
    print(f"   정답: {prompt['answer']}")
    print(f"   그룹 크기: {group_size}")
    
    # 1. G개 응답 생성 (시뮬레이션)
    print(f"\n   Step 1: {group_size}개 응답 생성")
    outputs = []
    for i in range(group_size):
        # 일부는 정답, 일부는 오답으로 시뮬레이션
        correct = random.random() > 0.4  # 60% 정답률
        if correct:
            answer = prompt['answer']
        else:
            answer = str(int(prompt['answer']) + random.choice([-2, -1, 1, 2]))
        outputs.append({"answer": answer, "correct": str(answer) == str(prompt['answer'])})
        status = "✅" if outputs[-1]["correct"] else "❌"
        print(f"      응답 {i+1}: {answer} {status}")
    
    # 2. 보상 계산
    print(f"\n   Step 2: 보상 계산")
    rewards = []
    for o in outputs:
        r = 1.0 if o["correct"] else 0.0
        rewards.append(r)
        print(f"      r = {r:.1f}")
    
    rewards = np.array(rewards)
    
    # 3. 그룹 내 정규화
    print(f"\n   Step 3: 그룹 내 정규화")
    mean_r = np.mean(rewards)
    std_r = np.std(rewards) + 1e-8
    advantages = (rewards - mean_r) / std_r
    
    print(f"      평균: {mean_r:.3f}, 표준편차: {std_r:.3f}")
    for i, (adv, r) in enumerate(zip(advantages, rewards)):
        direction = "↑ 강화" if adv > 0 else "↓ 약화" if adv < 0 else "→ 유지"
        print(f"      Â_{i+1} = ({r:.1f} - {mean_r:.3f}) / {std_r:.3f} = {adv:+.3f} ({direction})")
    
    # 4. 정책 업데이트 방향
    print(f"\n   Step 4: 정책 업데이트 방향")
    for i, (o, adv) in enumerate(zip(outputs, advantages)):
        if adv > 0:
            print(f"      응답 {i+1} ({o['answer']}): 확률 증가 ↑ (정답)")
        elif adv < 0:
            print(f"      응답 {i+1} ({o['answer']}): 확률 감소 ↓ (오답)")
    
    return advantages

# 시뮬레이션 실행
random.seed(42)
prompts = [
    {"question": "15 + 27 = ?", "answer": "42"},
    {"question": "8 × 7 = ?", "answer": "56"},
]

for p in prompts:
    print("\n" + "=" * 50)
    simulate_grpo_step(p, group_size=4)

📌 GRPO 알고리즘 시뮬레이션


📝 프롬프트: 15 + 27 = ?
   정답: 42
   그룹 크기: 4

   Step 1: 4개 응답 생성
      응답 1: 42 ✅
      응답 2: 43 ❌
      응답 3: 41 ❌
      응답 4: 42 ✅

   Step 2: 보상 계산
      r = 1.0
      r = 0.0
      r = 0.0
      r = 1.0

   Step 3: 그룹 내 정규화
      평균: 0.500, 표준편차: 0.500
      Â_1 = (1.0 - 0.500) / 0.500 = +1.000 (↑ 강화)
      Â_2 = (0.0 - 0.500) / 0.500 = -1.000 (↓ 약화)
      Â_3 = (0.0 - 0.500) / 0.500 = -1.000 (↓ 약화)
      Â_4 = (1.0 - 0.500) / 0.500 = +1.000 (↑ 강화)

   Step 4: 정책 업데이트 방향
      응답 1 (42): 확률 증가 ↑ (정답)
      응답 2 (43): 확률 감소 ↓ (오답)
      응답 3 (41): 확률 감소 ↓ (오답)
      응답 4 (42): 확률 증가 ↑ (정답)


📝 프롬프트: 8 × 7 = ?
   정답: 56
   그룹 크기: 4

   Step 1: 4개 응답 생성
      응답 1: 56 ✅
      응답 2: 56 ✅
      응답 3: 58 ❌
      응답 4: 54 ❌

   Step 2: 보상 계산
      r = 1.0
      r = 1.0
      r = 0.0
      r = 0.0

   Step 3: 그룹 내 정규화
      평균: 0.500, 표준편차: 0.500
      Â_1 = (1.0 - 0.500) / 0.500 = +1.000 (↑ 강화)
      Â_2 = (1.0 - 0.500) / 0.500 = +1.000 (↑ 강화)
      Â_3 = (0.0 - 0.500) /

---
## 3️⃣ Chain-of-Thought 추론 패턴 분석

GRPO 학습 후 모델이 어떤 추론 패턴을 보이는지 분석합니다.

In [18]:
# GRPO 학습의 영향 분석
print("=" * 60)
print("📌 GRPO 학습 영향 분석")
print("=" * 60)

print("""
📊 GRPO 학습으로 기대되는 변화:

1️⃣ 정답률 향상
   • 정답에 대한 보상(+1.0)으로 정확한 계산 능력 강화
   • 오답 패턴의 확률 감소

2️⃣ 형식 준수율 향상
   • 형식 보상(+0.5)으로 '정답: [숫자]' 패턴 학습
   • 일관된 출력 형식 유지

3️⃣ 풀이 과정 생성
   • 풀이 보상(+0.25)으로 단계적 설명 능력 향상
   • Chain-of-Thought 패턴 발현 가능

4️⃣ DeepSeek R1과의 차이
   • 규모: R1은 671B, 우리는 1.5B (약 450배 차이)
   • 데이터: R1은 수십만 문제, 우리는 100문제
   • 그룹 크기: R1은 G=64, 우리는 G=4
   • 학습 시간: R1은 수천 GPU-hour, 우리는 ~1시간
   
   → 하지만 GRPO의 핵심 원리는 동일합니다!
   → 더 많은 데이터와 더 큰 모델로 확장하면 성능이 크게 향상됩니다.
""")

📌 GRPO 학습 영향 분석

📊 GRPO 학습으로 기대되는 변화:

1️⃣ 정답률 향상
   • 정답에 대한 보상(+1.0)으로 정확한 계산 능력 강화
   • 오답 패턴의 확률 감소

2️⃣ 형식 준수율 향상
   • 형식 보상(+0.5)으로 '정답: [숫자]' 패턴 학습
   • 일관된 출력 형식 유지

3️⃣ 풀이 과정 생성
   • 풀이 보상(+0.25)으로 단계적 설명 능력 향상
   • Chain-of-Thought 패턴 발현 가능

4️⃣ DeepSeek R1과의 차이
   • 규모: R1은 671B, 우리는 1.5B (약 450배 차이)
   • 데이터: R1은 수십만 문제, 우리는 100문제
   • 그룹 크기: R1은 G=64, 우리는 G=4
   • 학습 시간: R1은 수천 GPU-hour, 우리는 ~1시간

   → 하지만 GRPO의 핵심 원리는 동일합니다!
   → 더 많은 데이터와 더 큰 모델로 확장하면 성능이 크게 향상됩니다.



---

## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

1️⃣ **DeepSeek R1 학습 파이프라인**: Cold Start SFT → Reasoning RL(GRPO) → Rejection Sampling + SFT → 종합 RL

2️⃣ **GRPO 알고리즘**: 그룹 내 상대 비교로 Value Model 없이 학습 → 메모리 효율적

3️⃣ **Rule-based Reward**: 정답 보상(+1.0) + 형식 보상(+0.5) — 검증가능 태스크에 적합

4️⃣ **자발적 CoT 발현**: "Aha Moment" — RL만으로 모델이 자기반성·재추론 패턴 학습

### 다음 세션 예고

- 🔜 **Session 20b**: Rejection Sampling + SFT 실습 — R1의 3단계 재현
- 🔜 **Session 21**: GRPOTrainer로 GRPO 직접 학습 — R1의 2단계 재현

### 💡 분석 과제
1. R1의 4단계 중 어떤 단계가 가장 비용이 큰지 추정
2. Rule-based reward가 잘 작동하는 태스크 vs 어려운 태스크 분류
3. (선택) R1 논문에서 보고된 정확도 곡선과 자발적 CoT 출현 시점 비교

---

In [ ]:
print("✅ Session 20 완료 — 다음은 Session 20b(RS 실습), Session 21(GRPO 실습)")